# Winter wheat yield — TorchCrop against GDHY (gridded)

**What this notebook does.** It bins the 10 km TorchCrop winter-wheat yield
simulation onto the 0.5° grid of GDHY (Iizumi & Sakai 2020), pairs the two
fields per grid cell and year, and reports the same skill metrics as
[`yield_evaluation.ipynb`](yield_evaluation.ipynb) — but spatially resolved
rather than collapsed to 23 national means.

**Why a second yield notebook.** CyBench evaluates at the level a country runs
its statistics service at; GDHY evaluates at the level the model actually
predicts. The two ask different questions and can disagree: a country-level
bias can hide entirely opposite errors in its western and eastern halves, and
this notebook is where that would show up.

**Three conventions worth knowing before reading any number:**

1. **Error is `simulated − observed`.** A negative bias means the model is low.
2. **GDHY is not an observation of the grid cell**, it downscales national and
   sub-national statistics with a satellite vegetation-index proxy and a crop
   mask. A cell's *level* is largely its country's statistic; what varies
   between neighbouring cells is mostly the proxy and the mask, so a spatial
   correlation partly measures agreement with the mask, and neighbouring cells
   are not independent samples. §7 separates the spatial and interannual
   signal for exactly this reason.
3. **No moisture conversion is applied**, for the same reason as the CyBench
   notebook: GDHY inherits the moisture basis of the national statistics behind
   it, TorchCrop reports grain dry matter, and both are compared as published.

All reusable code is in [`utils/`](utils/); this notebook is the workflow only.

## 0. Setup

In [ ]:
import logging
import sys
from pathlib import Path

# cropmodelling4eu is installed (pip install -e .), so the evaluation
# library is imported like any other package rather than off sys.path.

import numpy as np
import pandas as pd

from cropmodelling4eu.evaluation import config, doy, grid, metrics, plots, torchcrop
from cropmodelling4eu.evaluation.style import use_style

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s",
                    force=True)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

PALETTE = use_style("light")
config.ensure_output_dirs()

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 140)

print(f"TorchCrop run : {config.TORCHCROP_RUN_DIR}")
print(f"Outputs       : {config.OUTPUT_DIR}")
print(f"Grid          : {config.GRID_RES_DEG}° over {config.EUROPE_BBOX}")

## 1. Load the TorchCrop simulation and bin it onto the GDHY grid

Cells are averaged into whichever 0.5° cell their centre falls in
(`utils.grid.bin_cells`), unweighted — the export carries no per-cell wheat
area, the same limitation as the country-level notebook. A 0.5° cell is kept
only once it holds at least `config.MIN_CELLS_PER_GRIDCELL` simulated 10 km
cells, so a cell that is mostly sea or mostly non-cropland does not carry the
same weight as a fully covered inland one.

In [ ]:
sim = torchcrop.load_simulation(columns=[
    "SimplaceID", "year", "lon", "lat", "yield_t_ha", "biomass_g_m2", "max_lai",
    "days_to_maturity", "tranrf_mean", "nni_mean", "heat_stress_factor", "irri",
])
sim = grid.crop_to_bbox(sim, config.EUROPE_BBOX)
# Tagged onto every 10 km row (not just the binned means) so the diagnostics
# in §8 can group the run's own state variables by 0.5° cell too.
sim["grid_id"] = grid.grid_cell_id(*grid.snap_to_grid(sim["lon"], sim["lat"]))

sim_grid = grid.bin_cells(sim, {"yield_t_ha": False}, by=["year"])
print(f"{len(sim):,} cell-seasons on the 10 km grid -> "
      f"{len(sim_grid):,} cell-years on the {config.GRID_RES_DEG}° grid, "
      f"{sim_grid['grid_id'].nunique():,} distinct 0.5° cells")
sim_grid.describe().T

## 2. Load GDHY

One NetCDF per year, cropped to the European bounding box and stacked. `year`
is the file's own label; GDHY dates a crop by its harvest year, the same
convention `torchcrop.load_simulation` uses, so
`config.GDHY_HARVEST_YEAR_OFFSET` is 0 by default — checked in §3, not assumed.

In [ ]:
from cropmodelling4eu.evaluation import gdhy

print(f"GDHY crop: {config.GDHY_CROP} ({config.GDHY_ROOT})")
published = gdhy.available_years(crop=config.GDHY_CROP)
print(f"published years: {published[0]}-{published[-1]}")

obs_grid = gdhy.load_gdhy(crop=config.GDHY_CROP)
print(f"{len(obs_grid):,} cell-years, {obs_grid['grid_id'].nunique():,} cells, "
      f"{obs_grid['year'].min()}-{obs_grid['year'].max()}")
obs_grid["obs_yield"].describe()

## 3. Check the harvest-year convention

`torchcrop.load_simulation` labels a row by the calendar year the crop was
**harvested** in — a crop sown in autumn 1999 and maturing in mid-2000 is
`year = 2000`. If GDHY's file year meant something else (e.g. the sowing year),
pairing at offset 0 would silently compare each season against the wrong
year's weather. Pairing at three offsets and comparing the pooled RMSE is a
crude but honest check: the correct offset should not be worse than its
neighbours.

In [ ]:
offset_check = []
for offset in (-1, 0, 1):
    shifted = obs_grid.assign(year=(obs_grid["year"] + offset).astype("int16"))
    trial = grid.pair_gridded(sim_grid, shifted, ["grid_id", "year"])
    m = metrics.yield_metrics(trial["obs_yield"], trial["yield_t_ha"])
    offset_check.append({"offset": offset, "n": m["n"], "rmse": m["rmse"],
                         "bias": m["bias"], "pearson_r": m["pearson_r"]})

offset_table = pd.DataFrame(offset_check).set_index("offset")
print(f"config.GDHY_HARVEST_YEAR_OFFSET = {config.GDHY_HARVEST_YEAR_OFFSET}")
offset_table.round(3)

> Read this table for **stability, not a winner.** The correlation is
> dominated by the spatial pattern common to all three offsets (§7 shows
> exactly how much), so it barely moves. What would flag a wrong convention is
> a large RMSE or bias jump at the configured offset relative to its
> neighbours; a small, smooth change across all three is the expected shape
> when the offset is right and the residual signal is weak.

## 4. Pair the two sides

In [ ]:
paired = grid.pair_gridded(sim_grid, obs_grid, ["grid_id", "year"])
paired["residual"] = paired["yield_t_ha"] - paired["obs_yield"]

print(f"{len(paired):,} paired cell-years across {paired['grid_id'].nunique():,} "
      f"0.5° cells, {paired['year'].min()}-{paired['year'].max()}")
paired.head()

## 5. Pooled metrics

Pooled over every cell-year — dominated by the spatial pattern, as noted above.
`n` here is cell-years, not the 412 country-years of the CyBench notebook, so
the two RMSEs are not read against the same denominator.

In [ ]:
pooled = metrics.yield_metrics(paired["obs_yield"], paired["yield_t_ha"])
print("Pooled over every 0.5-degree cell-year:")
for key in metrics.YIELD_METRIC_ORDER:
    print(f"  {metrics.METRIC_LABELS[key]:>16s}  {pooled[key]:>8.2f}")

## 6. Figures — the raw field

In [ ]:
fig = plots.scatter_density(
    paired, "obs_yield", "yield_t_ha", stats=pooled,
    title=f"Gridded winter wheat yield, {paired['year'].min()}-{paired['year'].max()}",
    xlabel="GDHY observed yield (t ha$^{-1}$)",
    ylabel="TorchCrop simulated yield (t ha$^{-1}$)",
)
plots.save(fig, "gdhy_01_scatter_density")
fig

In [ ]:
cell_bias = paired.groupby(["grid_id", "lon", "lat"], as_index=False)["residual"].mean()

fig = plots.cell_map(
    cell_bias, "residual",
    title="Mean yield bias by 0.5° cell (simulated - observed)",
    cbar_label="Bias (t ha$^{-1}$)", diverging=True,
)
plots.save(fig, "gdhy_02_map_bias")
fig

This is the map the country-level notebook cannot draw. Compare it with
`outputs/figures/yield_07_map_bias.pdf` — if the two agree at the coastline
where a CyBench country boundary sits, the country-level bias was not hiding a
finer structure; if they disagree, this map is the more honest one.

In [ ]:
fig = plots.timeseries_pair(
    paired, "obs_yield", "yield_t_ha",
    title="Domain-mean winter wheat yield through time",
    ylabel="Yield (t ha$^{-1}$)",
)
plots.save(fig, "gdhy_03_timeseries")
fig

In [ ]:
fig = plots.residual_panels(
    paired, "residual", "obs_yield",
    title="Yield residuals (gridded)",
    ylabel="Residual (t ha$^{-1}$)",
    xlabel_obs="GDHY observed yield (t ha$^{-1}$)",
)
plots.save(fig, "gdhy_04_residuals")
fig

## 7. Spatial signal against interannual signal

Splitting a cell's yield into its long-term mean (spatial) and its deviation
from that mean in a given year (interannual, via `metrics.to_anomalies`)
answers the question §2's caveat raises directly: how much of the correlation
above is "the model puts high-yielding cells where GDHY does" against "the
model's good and bad years line up with GDHY's".

In [ ]:
anom = metrics.to_anomalies(paired, ["obs_yield", "yield_t_ha"])

spatial_mean = paired.groupby("grid_id")[["obs_yield", "yield_t_ha"]].mean()
spatial = metrics.yield_metrics(spatial_mean["obs_yield"], spatial_mean["yield_t_ha"])
interannual = metrics.yield_metrics(anom["obs_yield_anom"], anom["yield_t_ha_anom"])

decomposition = pd.DataFrame(
    {"Spatial (cell means)": spatial, "Interannual (anomalies)": interannual}
).T[list(metrics.YIELD_METRIC_ORDER)]
decomposition.round(3)

> If the spatial row's `pearson_r` is far from zero and the interannual row's
> is close to it, the model is reproducing *where* GDHY expects high yields but
> not *when* — its year-to-year variation is close to noise relative to GDHY's.
> That is a materially different statement from the pooled r in §5, which
> cannot tell the two apart.

In [ ]:
cell_r = (
    anom.groupby("grid_id")
    .apply(lambda g: metrics.yield_metrics(g["obs_yield"], g["yield_t_ha"])["pearson_r"]
           if len(g) >= 5 else np.nan, include_groups=False)
    .rename("interannual_r")
)
cell_pos = paired.groupby(["grid_id", "lon", "lat"], as_index=False).size().drop(columns="size")
cell_pos["interannual_r"] = cell_pos["grid_id"].map(cell_r)

share_positive = float((cell_r.dropna() > 0).mean())
print(f"{cell_r.notna().sum()} cells with >=5 paired years; "
      f"{100 * share_positive:.0f}% have a positive interannual correlation")

fig = plots.cell_map(
    cell_pos.dropna(subset=["interannual_r"]), "interannual_r",
    title="Interannual correlation per cell (>= 5 paired years)",
    cbar_label="Pearson r", diverging=True, vmax=1.0,
)
plots.save(fig, "gdhy_05_map_interannual_r")
fig

## 8. Diagnostics — the run's own state next to the bias

The same diagnostic the country-level notebook runs in its §9, regridded: the
run's state variables, averaged per 0.5° cell, correlated against that cell's
bias.

In [ ]:
state = (
    sim.groupby("grid_id", as_index=False)
    .agg(lon=("lon", "first"), lat=("lat", "first"),
         max_lai=("max_lai", "mean"), biomass=("biomass_g_m2", "mean"),
         days_to_maturity=("days_to_maturity", "mean"),
         water_stress=("tranrf_mean", "mean"), n_index=("nni_mean", "mean"),
         heat=("heat_stress_factor", "mean"),
         failed_share=("yield_t_ha", lambda s: float((s < 0.5).mean())))
)

state["obs_yield"] = state["grid_id"].map(spatial_mean["obs_yield"])
state["sim_yield"] = state["grid_id"].map(spatial_mean["yield_t_ha"])
state["bias"] = state["sim_yield"] - state["obs_yield"]
state = state.dropna(subset=["bias"])

print("Correlation of cell bias with the run's own state variables:")
print(state.corr(numeric_only=True)["bias"].drop(columns=[], errors="ignore")
      .drop("bias").round(2).sort_values().to_string())

fig = plots.cell_map(
    state, "max_lai",
    title="Simulated maximum leaf area index (mean over seasons)",
    cbar_label="Max LAI (m$^2$ m$^{-2}$)",
)
plots.save(fig, "gdhy_06_map_max_lai")
fig

## 9. Summary

In [ ]:
summary = pd.Series({
    "0.5-degree cells": paired["grid_id"].nunique(),
    "cell-years": len(paired),
    "years": f"{paired['year'].min()}-{paired['year'].max()}",
    "observed mean (t/ha)": round(pooled["obs_mean"], 2),
    "simulated mean (t/ha)": round(pooled["sim_mean"], 2),
    "bias (t/ha)": round(pooled["bias"], 2),
    "RMSE (t/ha)": round(pooled["rmse"], 2),
    "pooled Pearson r": round(pooled["pearson_r"], 2),
    "spatial Pearson r": round(spatial["pearson_r"], 2),
    "interannual Pearson r": round(interannual["pearson_r"], 2),
}, name="value")

paired.to_csv(config.TABLE_DIR / "gdhy_paired_cell_year.csv",
              index=False, float_format="%.4f")
decomposition.to_csv(config.TABLE_DIR / "gdhy_spatial_interannual.csv",
                     float_format="%.4f")
state.to_csv(config.TABLE_DIR / "gdhy_diagnostics_by_cell.csv", float_format="%.4f")
print(f"tables written to {config.TABLE_DIR}")
print(f"figures written to {config.FIGURE_DIR}")
summary.to_frame()

### What the numbers mean, and what they do not

* **The gridded bias map is the finer-grained twin of `yield_07_map_bias`.**
  Where the two agree, the country-level number was not hiding structure the
  country boundary happens to cut through; where they disagree, trust this one.
* **The pooled correlation is not a skill score.** §7's spatial/interannual
  split is the number to quote for "does the model track a good year" — the
  pooled figure mixes that question with "does the model know where wheat
  yields well", which GDHY answers largely from its own crop mask.
* **A single p-value for the pooled fit would be fiction.** Tens of thousands
  of cell-years are not independent samples; two neighbouring cells share most
  of their signal. No p-value is reported here for that reason — read the
  effect sizes only.
* **The moisture offset from the CyBench notebook still applies.** No
  conversion is applied here either, so part of any negative bias is expected
  before model error is counted.